# Week 09 - Hybrid Models

**Mandatory output of master plan section 11.9.**

## Hybrid A: Black-Scholes residual learning

$$\varepsilon_i = C^{(i)}_{market} - C^{(i)}_{BS}, \qquad x_i \to \varepsilon_i,
\qquad \hat{C}_{hybrid} = C_{BS} + \hat{\varepsilon}$$

Section 5.5 states the purpose: the task is reduced from *learn the complete
pricing function* to *learn only the systematic part of the model error*.

### The milestone, and the trap it points at

> The same option's implied volatility shall not be used to define the
> residual-learning benchmark.

If $C_{BS}$ were priced with the option's own implied volatility then, by the
definition of implied volatility, $C_{BS} = C_{market}$ and the residual is
identically zero. The model learns nothing, and the metrics look *perfect* -
which is what makes it dangerous rather than merely wrong.
`hybrid_models.residual_dataset()` refuses a benchmark that reproduces the
market price too closely, so the mistake raises instead of passing.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (11, 3.8), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

from hybrid_models import make_smile_dataset, HybridA, smile_sigma, residual_dataset
from linear_models import Standardiser, fit_ols, metrics
from neural_net import MLP
from tree_models import RandomForest
SEED, FEATS = 20260825, ["m","T","r","sigma_atm"]

## 1. A market where the answer is known

On real data the residual is whatever it is, and "the hybrid improved RMSE"
could mean it learned the model error or that it memorised noise. So we
manufacture a market: prices from a volatility **smile**, benchmark priced
with the flat at-the-money volatility - what a desk carrying one number per
expiry would quote.

In [ ]:
d = make_smile_dataset(n=20000, seed=SEED)
mm = np.linspace(-0.6, 0.6, 200)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(mm, 100*smile_sigma(mm, 0.25), lw=2)
ax[0].axhline(25, ls=":", color="black", label="flat ATM benchmark")
ax[0].set_xlabel("m"); ax[0].set_ylabel("volatility (%)")
ax[0].set_title("the smile the benchmark cannot see"); ax[0].legend(fontsize=8)
s = d.sample(4000, random_state=SEED)
ax[1].axhline(0, color="black", lw=1)
ax[1].scatter(s.m, s.epsilon, s=3, alpha=.3, color="#b5483d")
ax[1].set_xlabel("m"); ax[1].set_ylabel("C_market - C_BS")
ax[1].set_title("the residual Hybrid A learns")
plt.tight_layout(); plt.show()
print("mean |epsilon| = %.4f index points" % d.epsilon.abs().mean())

The residual is exactly zero at the money, where the smile touches the
benchmark, and grows into both wings. That shape is the thing to be learned.

## 2. The leakage guard

In [ ]:
leak = d.copy(); leak["sigma_own"] = d.sigma_true
try:
    residual_dataset(leak, sigma_bench_col="sigma_own")
    print("NO ERROR - the guard failed")
except ValueError as e:
    print("correctly refused:\n"); print(str(e)[:320])

## 3. Hybrid versus pure, same learners, same split

In [ ]:
rng = np.random.default_rng(SEED+3); perm = rng.permutation(len(d))
ntr, nva = int(.70*len(d)), int(.15*len(d))
tr, va, te = perm[:ntr], perm[ntr:ntr+nva], perm[ntr+nva:]
X, K = d[FEATS].to_numpy(), d.K.to_numpy()
eps, C_BS, C_mkt = d.epsilon.to_numpy(), d.C_BS.to_numpy(), d.C_market.to_numpy()
sc = Standardiser().fit(X[tr])

class _OLS:
    def fit(s,A,y): s.f=fit_ols(sc.transform(A),y); return s
    def predict(s,A): return s.f.predict(sc.transform(A))
class _RF:
    def fit(s,A,y): s.f=RandomForest(n_trees=40,max_depth=14,min_samples_leaf=5,seed=SEED).fit(A,y); return s
    def predict(s,A): return s.f.predict(A)
class _Net:
    val_y=None
    def __init__(s): s.n=MLP(4,hidden=(32,16),lr=3e-3,batch=256,epochs=600,seed=SEED,patience=80)
    def fit(s,A,y): s.n.fit(sc.transform(A),y,sc.transform(X[va]),_Net.val_y); return s
    def predict(s,A): return s.n.predict(sc.transform(A))

rows=[{"model":"Black-Scholes","kind":"benchmark",**metrics(C_mkt[te],C_BS[te])}]
for nm,mk in (("OLS",_OLS),("Random forest",_RF),("Neural net",_Net)):
    _Net.val_y = C_mkt[va]/K[va]
    p = mk().fit(X[tr], C_mkt[tr]/K[tr])
    rows.append({"model":nm,"kind":"pure ML",**metrics(C_mkt[te], p.predict(X[te])*K[te])})
    _Net.val_y = eps[va]/K[va]
    h = HybridA(mk()).fit(X[tr],K[tr],eps[tr])
    rows.append({"model":nm,"kind":"hybrid A",**metrics(C_mkt[te], h.predict(X[te],K[te],C_BS[te]))})
pd.DataFrame(rows)[["model","kind","MAE","RMSE","R2","max_abs_error"]].round(4)

**The hybrid beats the pure model with every learner.** Black-Scholes
already supplies the structure - discounting, the payoff shape, most of the
moneyness dependence - so the model only has to supply what the formula
misses. For OLS the difference is a factor of twelve: a linear map cannot
represent a price surface, but it can represent a smile correction.

## 4. The real-data path

`clean_options.py` then `build_real_dataset.py`. The benchmark volatility is
`sigma_atm_prev`: the at-the-money implied volatility of the **same expiry**
on the **previous** trading date. Observable before the date it is used on,
one number per expiry, and never the contract's own.

In [ ]:
rd = pd.read_csv("data/real_option_dataset.csv")
print("clean quotes      ", len(rd))
print("trading dates     ", rd.trade_date.nunique())
print("model-ready rows  ", int(rd.model_ready.sum()))
m = rd[rd.model_ready==True]
print("\nresidual: mean %+.2f, mean |eps| %.2f, %.0f%% one-sided"
      % (m.epsilon_BS.mean(), m.epsilon_BS.abs().mean(), 100*(m.epsilon_BS>0).mean()))

### What that residual is made of

The benchmark is ten days stale, so the residual mixes two things. Repricing
with *today's* at-the-money volatility separates them.

In [ ]:
from black_scholes import bs_call
Fd = (m.F_implied*np.exp(-m.r*m["T"])).to_numpy()
C_today = bs_call(Fd, m.K.to_numpy(), m["T"].to_numpy(), m.r.to_numpy(),
                  m.sigma_atm_today.to_numpy(), 0.0)
shape = np.abs(m.C_market.to_numpy()-C_today).mean()
total = m.epsilon_BS.abs().mean()
print("total  %6.2f" % total)
print("  level (stale vol) %6.2f  (%.0f%%)" % (total-shape, 100*(total-shape)/total))
print("  shape (the smile) %6.2f  (%.0f%%)" % (shape, 100*shape/total))
atm = rd[(rd.m.abs()<=.02)&rd.sigma_IV.notna()].groupby("trade_date").sigma_IV.median()
print("\nATM implied vol:"); print((100*atm).round(2).to_string())

**This is a finding about the sampling cadence, not about NIFTY.**

`sigma_atm_prev` goes stale at exactly the rate the sample is thinned. With
weekly files the benchmark is a week old, and the volatility *level* move over
that week swamps the smile shape - 63% against 37% here.

Master plan 5.5 asks the model to learn *"only the systematic part of the
model error"*. A level move over a week is the **unsystematic** part: roughly
a random walk, and no function of $[m, T, r]$ can predict it. Asking the
hybrid to learn it would be asking it to forecast volatility, which is a
different problem.

`sigma_hist` does not have this defect, because it comes from the daily NIFTY
close series, which is complete regardless of how often the option files are
sampled. That makes re-running `week1.py` a design requirement rather than a
missing column.

## 5. Where this leaves the checkpoint

> **Does residual learning reduce real-market Black-Scholes error?**

**On synthetic data, definitively yes** - a factor of 25 against the
benchmark, and the hybrid beats the pure model on every learner.

**On real data, not yet answerable.** Two trading dates, of which the first
cannot be model-ready because the benchmark needs a prior date. 531 usable
quotes is not a dataset and one date is not a chronological split.

What the two dates *do* establish is the composition of the residual, and
that is an argument the synthetic experiment could not have produced.